# Validate Ground Truth and Expose Stable Qrels

This notebook takes the LLM-labeled ground truth artifact created by notebook 1, validates it against a stratified human-labeled subset, computes agreement metrics, and exposes a stable qrels file only when the validation result is acceptable.

The workflow is:

1. Load query metadata, blinded annotation items, and LLM labels.
2. Sample a stratified human validation subset.
3. Create a human annotation sheet.
4. Compute quadratic weighted Cohen's kappa and the confusion matrix.
5. Export stable TREC qrels when the validation metrics are acceptable.

## Notebook Setup

This notebook reads artifacts from `groundtruth_outputs/annotation` and writes human validation artifacts to `groundtruth_outputs/human_validation`.

The stable qrels file is written to:

```text
groundtruth_outputs/qrels/qrels_500q_top50.txt
```

The default validation size is 30 queries, corresponding to approximately 1,500 query-document judgments if each query has 50 pooled candidates.

In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path
from typing import Optional

import pandas as pd


def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


FINALPROJECT_ROOT = find_finalproject_root()
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"
HUMAN_VALIDATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "human_validation"
QRELS_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "qrels"

QUERY_SET_PATH = NOTEBOOK_OUTPUT_DIR / "queries_500_title_as_query.csv"
BLINDED_ANNOTATION_PATH = ANNOTATION_OUTPUT_DIR / "blinded_annotation_items.jsonl"
LLM_LABELS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_labels.jsonl"
STABLE_QRELS_PATH = QRELS_OUTPUT_DIR / "qrels_500q_top50.txt"

HUMAN_VALIDATION_QUERY_COUNT = 30
SHUFFLE_RANDOM_SEED = 20260709
MIN_ACCEPTABLE_WEIGHTED_KAPPA = 0.60
ALLOW_SEVERE_DISAGREEMENTS = False

for directory_path in [HUMAN_VALIDATION_OUTPUT_DIR, QRELS_OUTPUT_DIR]:
    directory_path.mkdir(parents=True, exist_ok=True)

print("Output directory:", NOTEBOOK_OUTPUT_DIR)
print("LLM labels path:", LLM_LABELS_PATH)
print("Stable qrels path:", STABLE_QRELS_PATH)

## Load Ground Truth Build Artifacts

This section loads the outputs from notebook 1:

- query metadata,
- blinded annotation items,
- LLM relevance labels.

The validation notebook should not need RRF scores or method-origin metadata. It works only with the blinded judging input and the labels.

In [ ]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            if line.strip():
                records.append(json.loads(line))
    return records


queries = pd.read_csv(QUERY_SET_PATH)
blinded_annotation_items = load_jsonl_records(BLINDED_ANNOTATION_PATH)
llm_label_records = load_jsonl_records(LLM_LABELS_PATH)

llm_labels_dataframe = pd.DataFrame(llm_label_records)

print("Queries:", len(queries))
print("Blinded annotation items:", len(blinded_annotation_items))
print("LLM labels:", len(llm_labels_dataframe))
print("\nLLM relevance distribution:")
print(llm_labels_dataframe["relevance"].value_counts().sort_index())

## Step 1. Sample the Human Validation Queries

The human validation subset is sampled at the query level. Once a query is sampled, all 50 pooled candidate documents for that query should be labeled by a human annotator.

If `type_of_food` is available, sampling is balanced proportionally by that column. If not, the notebook uses reproducible random sampling.

In [ ]:
def sample_queries_for_human_validation(
    query_dataframe: pd.DataFrame,
    sample_size: int,
    random_seed: int,
) -> pd.DataFrame:
    """Sample validation queries, balanced by type_of_food when available."""
    if sample_size > len(query_dataframe):
        raise ValueError("sample_size cannot exceed the number of queries.")

    if "type_of_food" not in query_dataframe.columns:
        return query_dataframe.sample(n=sample_size, random_state=random_seed).reset_index(drop=True)

    random_generator = random.Random(random_seed)
    working_dataframe = query_dataframe.copy()
    working_dataframe["validation_stratum"] = working_dataframe["type_of_food"].astype(str)

    sampled_parts = []
    for _, stratum_group in working_dataframe.groupby("validation_stratum", dropna=False):
        stratum_fraction = len(stratum_group) / len(working_dataframe)
        stratum_sample_size = max(1, round(sample_size * stratum_fraction))
        stratum_sample_size = min(stratum_sample_size, len(stratum_group))
        sampled_parts.append(
            stratum_group.sample(
                n=stratum_sample_size,
                random_state=random_generator.randint(0, 10**9),
            )
        )

    sampled_queries = pd.concat(sampled_parts, ignore_index=True)
    if len(sampled_queries) > sample_size:
        sampled_queries = sampled_queries.sample(n=sample_size, random_state=random_seed)
    elif len(sampled_queries) < sample_size:
        remaining_queries = working_dataframe.loc[
            ~working_dataframe["query_id"].isin(sampled_queries["query_id"])
        ]
        additional_queries = remaining_queries.sample(
            n=sample_size - len(sampled_queries),
            random_state=random_seed,
        )
        sampled_queries = pd.concat([sampled_queries, additional_queries], ignore_index=True)

    return sampled_queries.sample(frac=1.0, random_state=random_seed).reset_index(drop=True)


human_validation_queries = sample_queries_for_human_validation(
    query_dataframe=queries,
    sample_size=HUMAN_VALIDATION_QUERY_COUNT,
    random_seed=SHUFFLE_RANDOM_SEED,
)

human_validation_queries_path = HUMAN_VALIDATION_OUTPUT_DIR / "human_validation_queries.csv"
human_validation_queries.to_csv(human_validation_queries_path, index=False, encoding="utf-8-sig")
print("Saved sampled validation queries to:", human_validation_queries_path)
human_validation_queries.head()

## Step 2. Create the Human Annotation Sheet

The human annotation sheet contains only blinded content fields and two blank columns:

- `human_relevance`
- `human_notes`

Human annotators must not see LLM labels, RRF rank, RRF score, method names, or original retrieval scores.

In [ ]:
def create_human_validation_sheet(
    blinded_annotation_items: list[dict],
    sampled_query_ids: set[int],
    output_path: Path,
) -> pd.DataFrame:
    """Create a CSV sheet for human annotation with blank relevance labels."""
    validation_rows = []
    for annotation_item in blinded_annotation_items:
        query_id = int(annotation_item["query_id"])
        if query_id not in sampled_query_ids:
            continue
        validation_rows.append(
            {
                "query_id": query_id,
                "doc_id": int(annotation_item["doc_id"]),
                "blinded_position": int(annotation_item["blinded_position"]),
                "query_text": annotation_item["query_text"],
                "recipe_title": annotation_item["recipe_title"],
                "recipe_type": annotation_item.get("recipe_type", ""),
                "ingredients": annotation_item.get("ingredients", ""),
                "normalized_ingredients": annotation_item.get("normalized_ingredients", ""),
                "cooking_steps": annotation_item.get("cooking_steps", ""),
                "human_relevance": "",
                "human_notes": "",
            }
        )
    validation_dataframe = pd.DataFrame(validation_rows)
    validation_dataframe.to_csv(output_path, index=False, encoding="utf-8-sig")
    return validation_dataframe


sampled_query_ids = set(human_validation_queries["query_id"].astype(int).tolist())
human_validation_sheet_path = HUMAN_VALIDATION_OUTPUT_DIR / "human_validation_annotation_sheet.csv"
human_validation_sheet = create_human_validation_sheet(
    blinded_annotation_items=blinded_annotation_items,
    sampled_query_ids=sampled_query_ids,
    output_path=human_validation_sheet_path,
)
print("Saved human validation sheet to:", human_validation_sheet_path)
print("Rows to annotate:", len(human_validation_sheet))
human_validation_sheet.head()

## Step 3. Compute Agreement Metrics

After the human annotation sheet has been completed, save it as:

```text
groundtruth_outputs/human_validation/human_validation_annotation_sheet_completed.csv
```

This section computes:

- quadratic weighted Cohen's kappa,
- confusion matrix,
- severe disagreement count for `0-vs-3` and `3-vs-0` errors.

In [ ]:
def build_confusion_matrix(
    human_labels: list[int],
    predicted_labels: list[int],
    labels: list[int],
) -> pd.DataFrame:
    """Build a confusion matrix with human labels as rows and predicted labels as columns."""
    label_to_index = {label: index for index, label in enumerate(labels)}
    matrix = [[0 for _ in labels] for _ in labels]
    for human_label, predicted_label in zip(human_labels, predicted_labels):
        matrix[label_to_index[int(human_label)]][label_to_index[int(predicted_label)]] += 1
    return pd.DataFrame(
        matrix,
        index=[f"human_{label}" for label in labels],
        columns=[f"llm_{label}" for label in labels],
    )


def quadratic_weighted_kappa(
    human_labels: list[int],
    predicted_labels: list[int],
    min_label: int = 0,
    max_label: int = 3,
) -> float:
    """Compute quadratic weighted Cohen's kappa without requiring sklearn."""
    if len(human_labels) != len(predicted_labels):
        raise ValueError("human_labels and predicted_labels must have the same length.")
    if not human_labels:
        raise ValueError("At least one label pair is required.")

    labels = list(range(min_label, max_label + 1))
    label_to_index = {label: index for index, label in enumerate(labels)}
    label_count = len(labels)
    observed_matrix = [[0.0 for _ in labels] for _ in labels]
    human_histogram = [0.0 for _ in labels]
    predicted_histogram = [0.0 for _ in labels]

    for human_label, predicted_label in zip(human_labels, predicted_labels):
        human_index = label_to_index[int(human_label)]
        predicted_index = label_to_index[int(predicted_label)]
        observed_matrix[human_index][predicted_index] += 1.0
        human_histogram[human_index] += 1.0
        predicted_histogram[predicted_index] += 1.0

    total_count = float(len(human_labels))
    expected_matrix = [
        [(human_histogram[i] * predicted_histogram[j]) / total_count for j in range(label_count)]
        for i in range(label_count)
    ]
    max_distance_squared = float((label_count - 1) ** 2)
    weighted_observed = 0.0
    weighted_expected = 0.0
    for i in range(label_count):
        for j in range(label_count):
            weight = ((i - j) ** 2) / max_distance_squared
            weighted_observed += weight * observed_matrix[i][j]
            weighted_expected += weight * expected_matrix[i][j]

    if weighted_expected == 0:
        return 1.0 if weighted_observed == 0 else 0.0
    return 1.0 - (weighted_observed / weighted_expected)


def evaluate_llm_human_agreement(
    llm_labels_path: Path,
    human_validation_completed_path: Path,
) -> dict:
    """Evaluate agreement between LLM labels and completed human labels."""
    llm_records = load_jsonl_records(llm_labels_path)
    llm_label_lookup = {
        (int(record["query_id"]), int(record["doc_id"])): int(record["relevance"])
        for record in llm_records
    }
    human_dataframe = pd.read_csv(human_validation_completed_path)
    human_dataframe = human_dataframe.dropna(subset=["human_relevance"])
    human_dataframe["human_relevance"] = human_dataframe["human_relevance"].astype(int)

    comparison_rows = []
    for _, row in human_dataframe.iterrows():
        key = (int(row["query_id"]), int(row["doc_id"]))
        if key in llm_label_lookup:
            comparison_rows.append(
                {
                    "query_id": key[0],
                    "doc_id": key[1],
                    "human_relevance": int(row["human_relevance"]),
                    "llm_relevance": int(llm_label_lookup[key]),
                }
            )
    comparison_dataframe = pd.DataFrame(comparison_rows)
    human_labels = comparison_dataframe["human_relevance"].astype(int).tolist()
    llm_labels = comparison_dataframe["llm_relevance"].astype(int).tolist()
    weighted_kappa = quadratic_weighted_kappa(human_labels, llm_labels, min_label=0, max_label=3)
    confusion_matrix = build_confusion_matrix(human_labels, llm_labels, labels=[0, 1, 2, 3])
    severe_disagreement_count = int(
        ((comparison_dataframe["human_relevance"] == 0) & (comparison_dataframe["llm_relevance"] == 3)).sum()
        + ((comparison_dataframe["human_relevance"] == 3) & (comparison_dataframe["llm_relevance"] == 0)).sum()
    )
    is_acceptable = weighted_kappa >= MIN_ACCEPTABLE_WEIGHTED_KAPPA and (
        ALLOW_SEVERE_DISAGREEMENTS or severe_disagreement_count == 0
    )
    return {
        "comparison_dataframe": comparison_dataframe,
        "weighted_kappa": weighted_kappa,
        "confusion_matrix": confusion_matrix,
        "severe_disagreement_count": severe_disagreement_count,
        "is_acceptable": is_acceptable,
    }


# Run this after the completed human validation CSV exists.
# completed_sheet_path = HUMAN_VALIDATION_OUTPUT_DIR / "human_validation_annotation_sheet_completed.csv"
# agreement_result = evaluate_llm_human_agreement(
#     llm_labels_path=LLM_LABELS_PATH,
#     human_validation_completed_path=completed_sheet_path,
# )
# print("Quadratic weighted Cohen's kappa:", agreement_result["weighted_kappa"])
# print("Severe 0-vs-3 disagreement count:", agreement_result["severe_disagreement_count"])
# print("Is acceptable:", agreement_result["is_acceptable"])
# display(agreement_result["confusion_matrix"])

## Step 4. Expose Stable Ground Truth Qrels

Only run this section after the agreement metrics are acceptable. The exposed file is the stable qrels file used by notebook 3.

Stable qrels format:

```text
query_id 0 doc_id relevance
```

In [ ]:
def export_trec_qrels_from_llm_labels(
    llm_labels_path: Path,
    qrels_output_path: Path,
    include_zero_relevance: bool = True,
) -> pd.DataFrame:
    """Export LLM labels to TREC qrels format."""
    llm_records = load_jsonl_records(llm_labels_path)
    qrels_rows = []
    for record in llm_records:
        relevance = int(record["relevance"])
        if relevance == 0 and not include_zero_relevance:
            continue
        qrels_rows.append(
            {
                "query_id": int(record["query_id"]),
                "iteration": 0,
                "doc_id": int(record["doc_id"]),
                "relevance": relevance,
            }
        )
    qrels_dataframe = pd.DataFrame(qrels_rows).sort_values(["query_id", "doc_id"]).reset_index(drop=True)
    qrels_dataframe.to_csv(qrels_output_path, sep=" ", header=False, index=False, encoding="utf-8")
    return qrels_dataframe


def summarize_qrels(qrels_dataframe: pd.DataFrame) -> pd.DataFrame:
    """Summarize qrels label distribution overall and per query."""
    overall_distribution = qrels_dataframe["relevance"].value_counts().sort_index().rename("count")
    per_query_counts = qrels_dataframe.groupby(["query_id", "relevance"]).size().unstack(fill_value=0)
    print("Overall relevance distribution:")
    print(overall_distribution)
    print("\nNumber of queries:", qrels_dataframe["query_id"].nunique())
    print("Total judged pairs:", len(qrels_dataframe))
    print("Judged pairs per query summary:")
    print(qrels_dataframe.groupby("query_id").size().describe())
    return per_query_counts


# Run only if agreement_result["is_acceptable"] is True.
# if agreement_result["is_acceptable"]:
#     qrels_dataframe = export_trec_qrels_from_llm_labels(
#         llm_labels_path=LLM_LABELS_PATH,
#         qrels_output_path=STABLE_QRELS_PATH,
#         include_zero_relevance=True,
#     )
#     per_query_qrels_summary = summarize_qrels(qrels_dataframe)
#     print("Saved stable qrels to:", STABLE_QRELS_PATH)
# else:
#     raise RuntimeError("Agreement metrics are not acceptable. Do not expose stable qrels yet.")